# Circle Packing — ShinkaEvolve Launcher

**Task**: Pack 26 non-overlapping circles inside the unit square `[0,1]²`  
**Objective**: maximise the **sum of radii** (best known ≈ 2.635 for n=26)  

### In this notebook

1. Checks Shinka imports and OpenRouter API key
2. Configures and launches the ShinkaEvolve run
3. Inspects results with plots and the best solution
4. Provides instructions for scaffolding other tasks for ShinkaEvolve

### Before getting started

Make sure that you've already completed the following.

-   Created and activated a virtual environment where ShinkaEvolve is installed.

-   Added a `.env` file with your OpenRouter API key to the root of this project.

Please see `README.md` at the root of this project for instructions on how to do if you've not already completed these steps! Finally,

-   If you are using Jupyterlab to edit this notebook in your web browser, make sure you've started your Jupyter server in the virtual environment

-   If you are editing this notebook in VSCode, make sure to select the Python kernel associated with the environment that you've created. It should say `Tutorial_Shinka (<version>)` with a Python executable located at `.venv/bin/python`.

## 1. Check imports and API keys

In [1]:
import sys
import logging
import warnings
import dotenv
import os
from pathlib import Path

# Suppress third-party warnings before importing shinka
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

# Check if ShinkaEvolve is importable
try:
    import shinka
except ImportError:
    print("shinka not found, make sure to open this notebook with the ShinkaEvolve environment active")

# Derive the venv activate script from the current Python interpreter.
# This works regardless of where the venv or the experiment folder lives.
activate_path = str(Path(sys.executable).parent / "activate")
print(f"activate_path  : {activate_path}")

# Find and load the .env file (for OPENROUTER_API_KEY)
env_path = dotenv.find_dotenv()
assert env_path, ".env not found, please add it to the root of this project."
dotenv.load_dotenv()

if os.environ.get("OPENROUTER_API_KEY"):
    print("OPENROUTER_API_KEY: OK")
else:
    print("WARNING: OPENROUTER_API_KEY not set — add it to .env file")

# Suppress the "Waiting for evaluation slot" message, which fires every 0.5s
class _SuppressWaitingFilter(logging.Filter):
    def filter(self, record):
        return "Waiting for evaluation slot" not in record.getMessage()

logging.getLogger("shinka.core.async_runner").addFilter(_SuppressWaitingFilter())

activate_path  : /Users/mp2557/Documents/Research/ShinkaEvolve/.venv/bin/activate
OPENROUTER_API_KEY: OK


## 2. Test the evaluator

Verify `evaluate.py` runs correctly on `initial_program.py` before launching evolution.

In [2]:
import evaluate
import initial_program

# Test the initial program for circle packing
output = initial_program.run_packing()

# Check if the the program outputs a valid circle packing
valid, msg = evaluate.validate_packing(output)

# Assert check
assert valid, f"Smoke test failed: {msg}"
print(f"Smoke test: PASSED  (sum_radii={output[2]:.6f})")

Smoke test: PASSED  (sum_radii=1.821228)


## 3. Configure Shinka evolution

In [3]:
import datetime as dt
from shinka.core import ShinkaEvolveRunner, EvolutionConfig
from shinka.database import DatabaseConfig
from shinka.launch import LocalJobConfig

# ── Task-specific parameters ──
TASK_SYS_MSG = """
You are an expert mathematician specialising in circle packing and computational geometry.
The best known result for the sum of radii when packing 26 circles in a unit square is 2.635.

Key directions to explore:
1. Variable-sized circles — the optimal packing uses circles of different radii.
2. Corner and edge placement — small circles fit efficiently near walls.
3. Hexagonal-grid initialisation combined with local optimisation (e.g. scipy L-BFGS-B / SLSQP).
4. Joint optimisation of centre positions AND radii as a single NLP.
5. Simulated annealing or basin-hopping for global search.
6. Greedy algorithms: place circles one at a time maximising contribution to sum.
7. Literature-inspired layouts for specific n (e.g. Packomania database).

Constraints:
- construct_packing() must return (centers, radii) with centers.shape=(26,2) and radii.shape=(26,).
- run_packing() (the fixed interface) calls construct_packing() and returns (centers, radii, sum_radii).
- All circles must lie strictly inside [0,1]^2 and must not overlap.
- HIGHER sum of radii = BETTER score.

Be creative. Try to beat 2.635."""

experiment_name = "circpack_" + dt.datetime.now().strftime("%Y%m%d_%H%M%S")
RESULTS_DIR = "results/" + experiment_name
print(f"Results dir    : {RESULTS_DIR}")

# ── EvoConfig parameters ──

LLM_MODELS = ["openrouter/anthropic/claude-haiku-4-5",
              "openrouter/openai/gpt-5.1-codex-mini",
              "openrouter/openai/gpt-5-nano",
              "openrouter/google/gemini-3-flash-preview"]

NUM_GENERATIONS = 10

# ── Other LLM parameters ── not too important for now but must be overwritten for OpenRouter keys (default is OpenAI)

META_LLM_MODELS = ["openrouter/openai/o4-mini"]
NOVELTY_LLM_MODELS = ["openrouter/openai/o4-mini"]
EMBEDDING_MODEL = "openrouter/openai/text-embedding-3-small"


# ── DBConfig parameters ──

NUM_ISLANDS = 2

###

evo_config = EvolutionConfig(task_sys_msg=TASK_SYS_MSG,
                            results_dir=RESULTS_DIR,
                            init_program_path="initial_program.py",
                            llm_models=LLM_MODELS,
                            num_generations=NUM_GENERATIONS,
                            meta_llm_models=META_LLM_MODELS,
                            novelty_llm_models=NOVELTY_LLM_MODELS,
                            embedding_model=EMBEDDING_MODEL)

db_config   = DatabaseConfig(num_islands=NUM_ISLANDS)

# activate_script ensures the subprocess uses the venv's Python (where shinka is installed)
job_config = LocalJobConfig(eval_program_path="evaluate.py",
                            activate_script=activate_path)

# conda version
# job_config = LocalJobConfig(eval_program_path="evaluate.py",
#                             conda_env="shinka")

Results dir    : results/circpack_20260420_093044


## 4. Launch evolution

In [4]:
from time import perf_counter

runner = ShinkaEvolveRunner(
    evo_config=evo_config,
    job_config=job_config,
    db_config=db_config,
    verbose=True,
)

tic = perf_counter()
await runner.run_async()
toc = perf_counter()

print(f"Evolution completed in {toc - tic:.1f} s")
print(f"Results saved to: {runner.results_dir}")

  @@@@@@@@@@@@@@@@@@@@@      ░██████╗██╗░░██╗██╗███╗░░██╗██╗░░██╗░█████╗░
  @                   @      ██╔════╝██║░░██║██║████╗░██║██║░██╔╝██╔══██╗
  @          @        @      ╚█████╗░███████║██║██╔██╗██║█████═╝░███████║
  @    @@   @@  @@    @      ░╚═══██╗██╔══██║██║██║╚████║██╔═██╗░██╔══██║
  @   @     @    @@   @      ██████╔╝██║░░██║██║██║░╚███║██║░╚██╗██║░░██║
  @    @@  @    @     @      ╚═════╝░╚═╝░░╚═╝╚═╝╚═╝░░╚══╝╚═╝░░╚═╝╚═╝░░╚═╝
  @        @          @      @@@@@@@@@@@@@@@
  @                   @   @@                 @@@@@
  @@@@@@@@@@@@@@@@@@@@ @@                       @  @@                 █▀▀
                      @                          @@  @                ██▄
                    @      @@                      @  @@
                   @       @         @              @   @             █░█
                   @                 @               @  @             ▀▄▀
                     @@@@@          @     @           @  @
                      @            @          @ 

2026-04-20 09:31:00 - shinka.core.async_runner - INFO - 🖥️  System resources detected:

2026-04-20 09:31:00 - shinka.core.async_runner - INFO -    • CPU cores: 8

2026-04-20 09:31:00 - shinka.core.async_runner - INFO -    • Memory: 16.0 GB

2026-04-20 09:31:00 - shinka.core.async_runner - INFO - 🔧 Concurrency settings:

...

2026-04-20 09:35:17 - shinka.llm.llm - INFO - ==> QUERYING: ['openrouter/openai/gpt-5-nano', '0.5', '16384']

2026-04-20 09:35:17 - httpx - INFO - HTTP Request: POST https://openrouter.ai/api/v1/responses "HTTP/1.1 200 OK"

2026-04-20 09:35:28 - shinka.database.async_dbase - INFO - 🔧 Closing async database with monitoring...

2026-04-20 09:35:28 - shinka.database.async_dbase - INFO - Async database closed

CancelledError: 

## 5. Web UI for evolution visualization - run this while evolution is running or after it is complete

In a separate Terminal, activate the ShinkaEvolve virtual environment and run the following command from the `TASK_DIR` directory:

`shinka_visualize --db results --port 8000 --open`

This will open the WebUI with a lot of information about the evolution.

**Warning**: there is often a bug when launching the Web UI, where the browser tab opens but the database is not loaded. To resolve it, click on `Dashboard` on the top left corner and the list of available databases should appear.

## 6. Inspect results - run this once the evolution is complete

In [ ]:
import matplotlib.pyplot as plt
from shinka.utils import load_programs_to_df
from shinka.plots import plot_lineage_tree, plot_evals_performance

results_root = Path("results") / experiment_name
# results_root = Path(runner.results_dir)

# The db may sit directly in results_root or one level up (shinka convention)
db_candidates = [
    results_root / "programs.sqlite"
    # results_root / results_root / "evolution_db.sqlite",
    # Path("evolution_db.sqlite"),
]
db_path = next((p for p in db_candidates if p.exists()), None)
assert db_path is not None, "Could not find evolution_db.sqlite"

df = load_programs_to_df(str(db_path))
print(f"Loaded {len(df)} programs from database.")

fig, axs = plt.subplots(1, 2, figsize=(28, 10), gridspec_kw={"width_ratios": [1, 1.5]})
fig.suptitle("Circle Packing v2 — ShinkaEvolve", fontsize=22, weight="bold")

plot_evals_performance(df, "Score over generations", fig, axs[0])
plot_lineage_tree(df, "Lineage tree", fig, axs[1])

plt.tight_layout()
plt.show()

## 7. Load and visualise the best solution

In [ ]:
import importlib.util
import numpy as np

best_program = results_root / "best" / "main.py"
assert best_program.exists(), f"Best program not found at {best_program}"

spec = importlib.util.spec_from_file_location("best_program", best_program)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

centers, radii, sum_radii = mod.run_packing()
print(f"Sum of radii (best): {sum_radii:.6f}  (target: 2.635)")
print(f"Min radius: {radii.min():.4f} | Max radius: {radii.max():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.set_title(f"Best packing — sum of radii = {sum_radii:.4f}", fontsize=13)
ax.add_patch(plt.Rectangle((0, 0), 1, 1, fill=False, edgecolor="black", lw=2))

cmap = plt.cm.tab20
for i, ((x, y), r) in enumerate(zip(centers, radii)):
    circ = plt.Circle((x, y), r, color=cmap(i % 20), alpha=0.6, linewidth=0.5, edgecolor="black")
    ax.add_patch(circ)
    ax.text(x, y, str(i), ha="center", va="center", fontsize=6)

plt.tight_layout()
plt.show()

---

# 9. Making your own experiment

This notebook works is specific for the circle packing task, but it can serve as a blueprint for making other tasks. Start by creating a separate folder with the following ingredients:

1. `initial_program.py`: This must contain the generator code for your task inside the `EVOLVE-BLOCK-START` / `EVOLVE-BLOCK-END` markers. Outside these you must add a `run_task()` function that returns the constructed object, its score, and possibly other statistics you might want to keep track of. You can also add any auxiliary functions you want here.
2. `evaluate.py`: This must contain a `validate_task(run_output)` function that must validate the correctness of the output of the `run_task()` function in the program of interest. It must also contain an `aggregate_metrics(results, results_dir)` function that returns a dictionary of evaluation metrics associated to the program. The most important one is `combined_score`, which is the score that is ultimately used to judge the quality of the program.
3. `shinka_launch.ipynb`: This one will be much closer to the current notebook, the only parameter that is task-specific is `TASK_SYS_MSG` in the `EvolutionConfig` instance. Another one I like to change is the name of the results databases, but it's not required. The visualization code at the end of this notebook is also specific for the circle packing task, but it is only used for a posteriori analysis of the solution found, not for running Shinka.

## Agentic usage

We can Claude Code (or other coding agents such as Codex) to help generating new shinka tasks. This is particularly useful for a first pass at the `initial_program.py` and `evaluate.py` files, mostly to create a skeleton that we can refine manually. I recommend copying this notebook for the actual `ShinkaEvolveRunner`, but coding agents can be useful for adding some context to the `TASK_SYS_MSG` as well.

To see how this work, see the the [Github repo at this link](https://sakanaai.github.io/ShinkaEvolve/agentic_usage/). The main part is Sections 1-4, once the task is scaffolded we can just follow the instructions in this notebook for running the evolution and visualizing progress.

---

# 10. Hyperparameters for ShinkaEvolve

There are many hyperparameters that can greatly affect the performance of ShinkaEvolve, and the best results will come from exploring different configurations, rather than a single run.

For a full list of hyperparameters see the [Github repo at this link](https://sakanaai.github.io/ShinkaEvolve/configuration/).

I will add more comments later on which hyperparameters are more likely to have an impact on the experiment.